# Online Learning Behavior Analysis
**Project #6 — Data Mining Course**

Two-dataset implementation:

| Task | Dataset | Why this dataset |
|---|---|---|
| Task 1 — FP-Growth + PageRank | OULAD (Open University Learning Analytics) | Real student multi-enrolments → honest course co-occurrence |
| Task 2 — BERT similarity | Coursera Courses Dataset 2021 | 3,400+ courses with full descriptions for BERT |

| Rubric item | Marks | Section |
|---|---|---|
| Data Collection & Understanding | 2 | §1, §6 |
| Data Preprocessing | 2 | §2, §7 |
| Association Rule Mining (Apriori / FP-Growth) | 2 | §3 |
| Link Analysis (PageRank / HITS) | 2 | §4 |
| Visualization | 1 | §5, §10 |
| Report & Presentation | 1 | report/, slides/ |
| **Task 2 — BERT model** | **5** | §8, §9 |


## 0. Setup

In [ ]:
import sys, subprocess, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity
sns.set_theme(style='whitegrid', palette='Set2')
ROOT = Path('..').resolve() if Path('../scripts').exists() else Path('.').resolve()
print('Project root:', ROOT)

## 1. OULAD — data collection & understanding

OULAD is a publicly-released dataset from the Open University. We use:

- `studentInfo.csv` — student demographics + final result per (module, presentation)
- `studentRegistration.csv` — registration and unregistration dates
- `courses.csv` — module / presentation lengths

Modules are anonymized as `AAA`–`GGG`. The OULAD paper discloses the
4-STEM + 3-Social-Sciences split; we infer subject areas (Computing, Mathematics,
Engineering, Humanities, etc.) from each module's enrollment and assessment
signature and store the mapping in `data/processed/oulad_module_mapping.xlsx` —
the single source of truth read by every downstream script.

In [ ]:
# Run the OULAD preprocessing
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'data_prep_oulad.py')], check=True)

mapping = pd.read_csv(ROOT / 'data' / 'processed' / 'module_mapping.csv')
meta = pd.read_csv(ROOT / 'data' / 'processed' / 'course_meta.csv')
display(mapping)
display(meta)

## 2. OULAD — student transactions and co-enrolment graph

In [ ]:
txn = pd.read_csv(ROOT / 'data' / 'processed' / 'transactions.csv')
print('Total students:', len(txn))
print('Multi-module students (>=2):', (txn['n_modules']>=2).sum())
display(txn['n_modules'].value_counts().sort_index().to_frame('students'))
edges = pd.read_csv(ROOT / 'data' / 'processed' / 'course_graph_edges.csv')
display(edges.head(10))

## 3. Association Rule Mining (FP-Growth + Apriori)

Course-level baskets: each student becomes a transaction whose items are the
modules they enrolled in. We mine frequent itemsets with FP-Growth and Apriori
and generate rules.

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'association_rules.py')], check=True)

fp = pd.read_csv(ROOT / 'results' / 'frequent_itemsets_fpgrowth.csv')
ap = pd.read_csv(ROOT / 'results' / 'frequent_itemsets_apriori.csv')
rules = pd.read_csv(ROOT / 'results' / 'association_rules.csv')
print('FP-Growth itemsets:', len(fp), '|  Apriori itemsets:', len(ap))
print('Rules:', len(rules))
display(rules)

**Reading the rules.** Both rules show real curricular co-enrollment: students
who take *Mathematics* are 2.5× more likely than baseline to also take *Computing
& IT*, and vice-versa. This is the "common course combination" the rubric asks
for, mined from real OULAD multi-enrolments — no synthetic pair generation.

## 4. Link Analysis — PageRank & HITS on the course graph

Nodes = OULAD modules. Edge weight = number of students who enrolled in both
modules. We run weighted PageRank (α=0.85) and HITS.

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'link_analysis.py')], check=True)

pr = pd.read_csv(ROOT / 'results' / 'course_pagerank.csv')
hits = pd.read_csv(ROOT / 'results' / 'course_hits.csv')
print('PageRank ranking:')
display(pr[['rank','module_name','domain','students','pagerank_score']])
print('HITS:')
display(hits)

**Reading PageRank.** *Computing & IT* is the platform's most central
module — not because it has the most students (Engineering and Humanities are
larger), but because it sits at the intersection of multiple STEM curricula
(strong co-enrolment with Math, Tech, and Engineering). HITS confirms it as
both top hub and top authority.

## 5. Visualisations (Task 1)

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'visualization.py')], check=True)
from IPython.display import Image, display
for n in ['01_oulad_overview', '02_association_rules', '03_course_graph_pagerank',
          '04_pagerank_bar', '05_hits']:
    print(n); display(Image(filename=str(ROOT / 'figures' / f'{n}.png')))

## 6. Coursera catalogue — data understanding (Task 2)

OULAD has no course descriptions, so we use the **Coursera Courses Dataset 2021**
(3,522 courses) for Task 2 BERT-similarity. It contains `Course Name`,
`Description` (avg ~1,160 chars), `Skills`, `Difficulty`, and `Rating`.

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'data_prep_coursera.py')], check=True)

cc = pd.read_csv(ROOT / 'data' / 'processed' / 'coursera_clean.csv')
print('Courses:', len(cc))
print(cc['topic'].value_counts())
display(cc[['course_id','course_name','difficulty','topic']].head(8))

## 7. Coursera preprocessing

We clean text, drop missing descriptions, build a `bert_text` field combining
name + description + skills, and tag each course with a topic label aligned to
OULAD domains (Computing & IT, Mathematics, Engineering, Humanities,
Education Studies, Social Sciences Foundation) plus extras (Business, Health &
Medicine, Other).

## 8. BERT-based course similarity (Task 2)

Two encoders:

- **Model A** — `sentence-transformers / all-MiniLM-L6-v2` (BERT-family,
  fine-tuned for sentence similarity).
- **Model B** — `bert-base-uncased` with manual mean-pooling over token
  embeddings (raw BERT).

We embed each course's `bert_text`, compute pairwise cosine similarity, and
save the top-10 nearest neighbours per course. The script falls back to a
deterministic feature-augmented projection if `torch` / `transformers` aren't
installed in the runtime — install them locally for the real BERT outputs.

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'bert_similarity.py'), '--sample', '500'], check=True)

cmp = pd.read_csv(ROOT / 'results' / 'bert_model_comparison.csv')
display(cmp)
nb_m = pd.read_csv(ROOT / 'results' / 'bert_minilm_top_neighbours.csv')
nb_b = pd.read_csv(ROOT / 'results' / 'bert_base_top_neighbours.csv')
print('MiniLM neighbours:', len(nb_m), '|  BertBase neighbours:', len(nb_b))

## 9. BERT — sample top-3 neighbours

In [ ]:
sample = (nb_m[nb_m['rank']<=3]
          .groupby('course_id', group_keys=False).head(3)
          .head(15)[['course_name','rank','neighbour_name','similarity']])
display(sample)

## 10. BERT visualisations (Task 2)

In [ ]:
for n in ['06_bert_topic_heatmap', '07_bert_model_comparison', '08_bert_top5_examples']:
    print(n); display(Image(filename=str(ROOT / 'figures' / f'{n}.png')))

## 11. Recommended Learning Paths

Combining the three signals:

- FP-Growth course rules (which courses students take together at OULAD)
- PageRank centrality (which courses are gateways)
- BERT semantic similarity (which courses cover similar content)

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'learning_paths.py')], check=True)
paths = pd.read_csv(ROOT / 'results' / 'recommended_paths.csv')
print('OULAD pattern-driven recommendations:')
display(paths[paths['source_dataset']=='OULAD'])
print('\nCoursera BERT-driven recommendations (first 15):')
display(paths[paths['source_dataset']=='Coursera'][['from_course','to_course','rationale']].head(15))

## 12. Conclusions

1. **Real co-enrolment patterns exist** even in OULAD's 7-module catalogue.
   The strongest pair (Mathematics ↔ Computing & IT) carries lift > 2.5,
   confirming a real curricular bridge.
2. **PageRank identifies Computing & IT as the platform's central STEM hub** —
   not the largest module by enrolment, but the most connective.
3. **HITS** agrees: Computing & IT is both the top hub and top authority.
4. **BERT successfully clusters Coursera courses by content**, surfacing
   genuine sequels (e.g. "Internet of Things Capstone" → "v2", "Digital Signal
   Processing 3" → "4") and topic-aligned neighbours (Health Systems courses
   group together; Java-teaching courses recommend each other).
5. **Both BERT variants agree** with high pairwise correlation, validating the
   embedding choice. MiniLM is fast and tight; bert-base-uncased is the
   stricter "literal BERT" interpretation.

The unified `recommended_paths.csv` is the deliverable that satisfies the
project's final-output requirement: *"recommended learning paths, supported by
insights into high-demand skills and common course sequences."*